In [ ]:
from dotenv import load_dotenv
import os
from typing import Dict, Any, Set, Optional
import pymongo
from pymongo import MongoClient
from pathlib import Path
from tqdm import tqdm
import underthesea
import json

In [ ]:



load_dotenv()
uri = os.getenv("MONGODB_URI")
print(uri)
mongo_client = MongoClient(uri)

In [ ]:
legal_sections = mongo_client["KB_PROPERTY_LAW"]["legal_sections"]
law_leaves = mongo_client["KB_PROPERTY_LAW"]["law_leaves"]

In [ ]:

def get_legal_section_by_id(doc_id: str) -> Optional[Dict[str, Any]]:
    """Lấy 1 doc từ legal_sections theo id"""
    doc = legal_sections.find_one({"_id": doc_id})
    return doc

In [ ]:
def get_all_parent_ids() -> Set[str]:
        """Lấy tất cả unique parent_id"""
        parents = legal_sections.aggregate([
            {"$match": {"parent_id": {"$ne": None}}},
            {"$group": {"_id": None, "parents": {"$addToSet": "$parent_id"}}}
        ])
        parents_list = list(next(parents, [{}])['parents'] or [])
        return set(parents_list)

In [ ]:
def get_and_build_law_leaves() -> int:
    """
    1. Get leaves (id NOT IN parents)
    2. Với mỗi leaf: Traverse parent tới type="điều" → Concat
    3. Upsert vào law_leaves
    """
    print("Caching all sections...")
    all_sections = {doc['id']: doc for doc in legal_sections.find({})}
    print(f"✓ Cached {len(all_sections)} sections")

    parents_set = get_all_parent_ids()
    leaf_ids = [doc_id for doc_id in all_sections if doc_id not in parents_set]
    print(f"✓ {len(leaf_ids)} leaves found")

    inserted = 0
    for leaf_id in tqdm(leaf_ids, desc="Building leaves"):
        leaf_doc = all_sections[leaf_id]

        # Traverse parents tới "điều"
        context_parts = []
        current_id = leaf_id
        while current_id in all_sections:
            doc = all_sections[current_id]
            if doc.get('type') == "điều":
                context_parts.append((doc.get('content') or '').strip())
                break  # Stop tại điều
            context_parts.append((doc.get('content') or '').strip())
            current_id = doc.get('parent_id')

        # Concat (điều trước, leaf sau)
        full_content = ". ".join(reversed(context_parts))

        # Tokens truncate optional
        tokens = underthesea.word_tokenize(full_content)
        if len(tokens) > 1024:
            full_content = ' '.join(tokens[:1024])

        law_leaf_doc = {
            'id': leaf_id,
            'full_content': full_content,
            'leaf_type': leaf_doc.get('type'),
            'full_path': leaf_doc.get('full_path'),
            'document_title': leaf_doc.get('document_title'),
            'so_hieu': leaf_doc.get('so_hieu'),
            'effective_date': leaf_doc.get('effective_date'),
            'parents_chain': context_parts[::-1],  # Reverse order
            'token_count': len(underthesea.word_tokenize(full_content)),
        }

        # Upsert (update nếu tồn tại)
        law_leaves.replace_one({'id': leaf_id}, law_leaf_doc, upsert=True)
        inserted += 1

    print(f"✓ Built & upserted {inserted} law_leaves!")
    return inserted

In [ ]:
print(len(get_all_parent_ids()))

In [ ]:
print(get_legal_section_by_id("85270b5d5ec8a7287bcb7e7f76827b2d4dc93806f125f15ad02ec03f84650433") )

In [ ]:
get_and_build_law_leaves()

In [ ]:
!pip install pymongo sentence-transformers faiss-cpu rank_bm25 underthesea numpy